In [1]:
import os
import sys
from pathlib import Path

# Where the notebook actually lives
NOTEBOOK_DIR = Path(os.getcwd())

# Repo root = parent of notebooks
REPO_DIR = NOTEBOOK_DIR.parent

print("Notebook directory:", NOTEBOOK_DIR)
print("Repo directory:", REPO_DIR)

# Force CityLearn from cloned repo (optional, see note below)
sys.path.insert(0, str(REPO_DIR))

import citylearn
print("✅ citylearn loaded from:", citylearn.__file__)

# Choose climate: TX or VT
CLIMATE = "VT"  # or "VT"

DATASET_DIR = REPO_DIR / "data" / "datasets" / f"annex96_ce1_{CLIMATE.lower()}_neighborhood"
SCHEMA_PATH = DATASET_DIR / "schema.json"

print("\nClimate:", CLIMATE)
print("Dataset directory:", DATASET_DIR, "| exists:", DATASET_DIR.exists())
print("Schema path:", SCHEMA_PATH, "| exists:", SCHEMA_PATH.exists())


Notebook directory: /home/kostas/school/Y2/Zoltan/annex96_common_exercise_1/notebooks
Repo directory: /home/kostas/school/Y2/Zoltan/annex96_common_exercise_1
✅ citylearn loaded from: /home/kostas/school/Y2/Zoltan/annex96_common_exercise_1/citylearn/__init__.py

Climate: VT
Dataset directory: /home/kostas/school/Y2/Zoltan/annex96_common_exercise_1/data/datasets/annex96_ce1_vt_neighborhood | exists: True
Schema path: /home/kostas/school/Y2/Zoltan/annex96_common_exercise_1/data/datasets/annex96_ce1_vt_neighborhood/schema.json | exists: True


In [2]:
import pandas as pd

district_target_df = pd.read_csv(DATASET_DIR / "district_target.csv")
district_target = district_target_df["district_load_target"].values

In [3]:
# system operations
import inspect
import os
import uuid
import warnings

# date and time
import datetime

# type hinting
from typing import Any

# User interaction
from ipywidgets import Button, HTML
from ipywidgets import Text, HBox, VBox

# data visualization
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from tqdm import tqdm

# data manipulation
from bs4 import BeautifulSoup
import math
import numpy as np
import pandas as pd
import random
import re
import requests
import simplejson as json

# cityLearn
from citylearn.agents.base import (
    BaselineAgent,
    Agent as RandomAgent
)
from citylearn.agents.rbc import HourRBC
from citylearn.agents.q_learning import TabularQLearning
from citylearn.citylearn import CityLearnEnv
from citylearn.data import DataSet
from citylearn.reward_function import RewardFunction, ComfortReward
from citylearn.wrappers import (
    NormalizedObservationWrapper,
    StableBaselines3Wrapper,
    TabularQLearningWrapper
)
from citylearn.agents.rbc import TemperatureBasedRBC
from citylearn.agents.rbc import PITemperatureController, PIDTemperatureController

# RL algorithms
from stable_baselines3 import SAC, PPO
import do_mpc

import copy

Couldn't import dot_parser, loading of dot files will not be possible.


2026-02-27 16:44:00.830387: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-27 16:44:00.845552: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-27 16:44:00.864568: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-27 16:44:00.870886: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-27 16:44:00.885745: I tensorflow/core/platform/cpu_feature_guar

In [4]:
import importlib, helpers
importlib.reload(helpers)
from helpers import select_buildings, select_simulation_period, get_kpis, plot_building_kpis, plot_district_kpis, plot_building_load_profiles, plot_district_load_profiles, plot_battery_soc_profiles, plot_comfort, plot_comfort_with_lstm_1step, make_env, run_env, plot_simulation_summary





In [5]:
# set all plotted figures without margins
plt.rcParams['axes.xmargin'] = 0
plt.rcParams['axes.ymargin'] = 0
%matplotlib inline

# ignore deprecation warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [6]:
import json
import pandas as pd

# Load schema.json
with open(SCHEMA_PATH, "r") as f:
    schema = json.load(f)


In [7]:
# List available buildings
schema["buildings"].keys()

dict_keys(['resstock-amy2018-2021-release-1-112208', 'resstock-amy2018-2021-release-1-147002', 'resstock-amy2018-2021-release-1-179247', 'resstock-amy2018-2021-release-1-199613', 'resstock-amy2018-2021-release-1-20199', 'resstock-amy2018-2021-release-1-216895', 'resstock-amy2018-2021-release-1-223581', 'resstock-amy2018-2021-release-1-245723', 'resstock-amy2018-2021-release-1-247942', 'resstock-amy2018-2021-release-1-319918', 'resstock-amy2018-2021-release-1-376570', 'resstock-amy2018-2021-release-1-408344', 'resstock-amy2018-2021-release-1-409896', 'resstock-amy2018-2021-release-1-411001', 'resstock-amy2018-2021-release-1-425540', 'resstock-amy2018-2021-release-1-4421', 'resstock-amy2018-2021-release-1-460412', 'resstock-amy2018-2021-release-1-467125', 'resstock-amy2018-2021-release-1-481052', 'resstock-amy2018-2021-release-1-485614', 'resstock-amy2018-2021-release-1-498771', 'resstock-amy2018-2021-release-1-525859', 'resstock-amy2018-2021-release-1-538628', 'resstock-amy2018-2021-rel

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

building_names = list(schema["buildings"].keys())

print(f"Found {len(building_names)} buildings")

for building_name in building_names:
    
    filename = schema["buildings"][building_name]["energy_simulation"]
    filepath = DATASET_DIR / filename
    
    print(f"\nProcessing: {building_name}")
    
    if not filepath.exists():
        print("  File not found. Skipping.")
        continue
    
    building_data = pd.read_csv(filepath)

    print(building_data["occupant_count"].value_counts().sort_index())
    
    # if "occupant_count" not in building_data.columns:
    #     print("  occupant_count column missing. Skipping.")
    #     continue
    
    # # Average occupancy per (month, hour)
    # occ_heatmap = (
    #     building_data
    #     .groupby(["month", "hour"])["occupant_count"]
    #     .mean()
    #     .unstack(level="hour")
    #     .reindex(columns=range(24), fill_value=0)
    # )
    
    # plt.figure(figsize=(14, 6))
    # plt.imshow(occ_heatmap, aspect="auto", origin="lower")
    # plt.colorbar(label="Average occupant count")
    
    # plt.xticks(np.arange(24))
    # plt.yticks(np.arange(len(occ_heatmap.index)), occ_heatmap.index)
    
    # plt.xlabel("Hour of day")
    # plt.ylabel("Month")
    # plt.title(f"Occupancy Heatmap\n{building_name}")
    # plt.tight_layout()
    # plt.show()

print("\nDone.")

Found 25 buildings

Processing: resstock-amy2018-2021-release-1-112208
occupant_count
0.0     768
1.0    3822
2.0    4170
Name: count, dtype: int64

Processing: resstock-amy2018-2021-release-1-147002
occupant_count
0.0     328
1.0    3021
2.0    5411
Name: count, dtype: int64

Processing: resstock-amy2018-2021-release-1-179247
occupant_count
0.0     360
1.0    2939
2.0    5461
Name: count, dtype: int64

Processing: resstock-amy2018-2021-release-1-199613
occupant_count
0.0     386
1.0    1473
2.0    6901
Name: count, dtype: int64

Processing: resstock-amy2018-2021-release-1-20199
occupant_count
0.0      51
1.0     655
2.0    2120
3.0    2661
4.0    3273
Name: count, dtype: int64

Processing: resstock-amy2018-2021-release-1-216895
occupant_count
0.0    2679
1.0    6081
Name: count, dtype: int64

Processing: resstock-amy2018-2021-release-1-223581
occupant_count
0.0    1087
1.0    7673
Name: count, dtype: int64

Processing: resstock-amy2018-2021-release-1-245723
occupant_count
0.0    1186
